<a href="https://colab.research.google.com/github/hamidrg20001379/rdhp-rdp-colab-experiment/blob/master/rdhp_rdp_fair_accuracy_ablation_all_seeds.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Fair accuracy and pair-correction ablation

This notebook runs seeds 42, 43, and 44 at matched global sparsities of 50% and 75%.

Primary methods are additive RDP, direct RDP, and reconstructed RDHP. Each primary method is fine-tuned with three independent training seeds for 5 and 20 epochs. A dense checkpoint control is fine-tuned with the same seeds and epoch counts.

A small pair-correction ablation evaluates shrinkage factors 0.0, 0.25, and 0.50 using distortion and immediate accuracy. The full-strength correction is intentionally excluded because the previous run was unstable.

## Requirements

- Colab GPU runtime.
- CIFAR-10 at `/content/drive/MyDrive/cifar10/cifar-10-batches-py`.
- Dense checkpoints at `/content/drive/MyDrive/rdhp_rdp_runs_seed_{42,43,44}/dense_resnet32.pt`.
- Secret `GITHUB_TOKEN` with read access to private `my-thesis` and its submodules.
- Secret `GITHUB_WRITE_TOKEN` with Contents read/write access to this experiment repository.

This is heavier than the previous benchmark because primary methods and the dense baseline are each repeated three times.

In [ ]:
from pathlib import Path
import copy
import getpass
import hashlib
import json
import os
import shutil
import subprocess
import sys
import time
from argparse import Namespace

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import torch
import torch.nn as nn

from google.colab import drive

OWNER = 'hamidrg20001379'
REPO_NAME = 'my-thesis'
REPO_URL = f'https://github.com/{OWNER}/{REPO_NAME}.git'
REPO = Path('/content') / REPO_NAME

token = None
try:
    from google.colab import userdata
    token = userdata.get('GITHUB_TOKEN')
except Exception:
    pass
if not token:
    token = getpass.getpass('GitHub token with private-repository read access: ')
if not token:
    raise RuntimeError('A GitHub token is required.')

if REPO.exists() and not (REPO / '.git').exists():
    raise RuntimeError(f'{REPO} exists but is not a Git checkout.')

askpass = Path('/tmp/github-askpass-full-network.sh')
askpass.write_text(
    '#!/bin/sh\n'
    'case "$1" in\n'
    "  *Username*) printf '%s\\n' x-access-token ;;\n"
    "  *Password*) printf '%s\\n' \"$GITHUB_TOKEN_FOR_ASKPASS\" ;;\n"
    "  *) printf '\\n' ;;\n"
    'esac\n'
)
askpass.chmod(0o700)
git_env = os.environ.copy()
git_env.update({
    'GIT_ASKPASS': str(askpass),
    'GIT_TERMINAL_PROMPT': '0',
    'GITHUB_TOKEN_FOR_ASKPASS': token,
})
try:
    if not REPO.exists():
        subprocess.run(['git', 'clone', '--recurse-submodules', REPO_URL, str(REPO)],
                       env=git_env, check=True)
    else:
        subprocess.run(['git', '-C', str(REPO), 'submodule', 'update', '--init', '--recursive'],
                       env=git_env, check=True)
finally:
    askpass.unlink(missing_ok=True)

EXPERIMENTS = REPO / 'working' / 'experiments'
sys.path.insert(0, str(EXPERIMENTS))

from rdp_resnet32_adapter import (
    accuracy,
    allocate_dp,
    apply_masks,
    build_curves as build_rdp_curves,
    load_checkpoint,
    load_resnet32,
    logits,
    make_loaders,
    model_stats,
    prunable_layers,
    seed_everything,
    train_epochs,
)
from rdhp_resnet32_adapter import build_rdhp_curves

DRIVE_ROOT = Path('/content/drive/MyDrive')
if not DRIVE_ROOT.exists():
    drive.mount('/content/drive')

DATA_ROOT = DRIVE_ROOT / 'cifar10'
DATASET_DIR = DATA_ROOT / 'cifar-10-batches-py'
for name in ['data_batch_1', 'data_batch_2', 'data_batch_3', 'data_batch_4', 'data_batch_5', 'test_batch']:
    if not (DATASET_DIR / name).exists():
        raise FileNotFoundError(f'Missing CIFAR-10 file: {DATASET_DIR / name}')

SEEDS = [42, 43, 44]
TARGET_SPARSITIES = [0.50, 0.75]
TRAIN_SAMPLES = 5000
CALIB_SAMPLES = 1024
TEST_SAMPLES = 1000
BATCH_SIZE = 128
CURVE_POINTS = 20
BUDGET_BINS = 5000
FISHER_PROBES = 2
FISHER_DAMPING = 1e-7
PAIR_COUNT = 24
PAIR_REFERENCE_RATIO = 0.50
FINETUNE_EPOCHS = [5, 20]
FINETUNE_SEED_OFFSETS = [1000, 2000, 3000]
PAIR_SHRINKAGE_FACTORS = [0.0, 0.25, 0.50]
PRIMARY_METHODS = ['additive_rdp', 'direct_rdp', 'rdhp']
LEARNING_RATE = 0.1
PUBLISH_RESULTS = True

OUTPUT_ROOT = DRIVE_ROOT / 'rdhp_rdp_fair_accuracy_ablation_all_seeds'
OUTPUT_ROOT.mkdir(parents=True, exist_ok=True)

print('Seeds:', SEEDS)
print('Target sparsities:', TARGET_SPARSITIES)
print('Device:', 'cuda' if torch.cuda.is_available() else 'cpu')
print('Fine-tuning seeds:', FINETUNE_SEED_OFFSETS)
print('Pair shrinkage factors:', PAIR_SHRINKAGE_FACTORS)
print('Output:', OUTPUT_ROOT)

In [ ]:
probe = load_resnet32()
layer_names = [
    name for name, module in probe.named_modules()
    if isinstance(module, (torch.nn.Conv2d, torch.nn.Linear))
]
layer_count = len(layer_names)
del probe

rng = np.random.default_rng(123)
all_pairs = [(a, b) for a in range(layer_count) for b in range(a + 1, layer_count)]
pair_indices = [all_pairs[i] for i in rng.choice(
    len(all_pairs), size=min(PAIR_COUNT, len(all_pairs)), replace=False
)]

print('Prunable layers:', layer_count)
print('Pair correction pairs:', pair_indices)
print('Pair names:')
for a, b in pair_indices:
    print((a, layer_names[a]), (b, layer_names[b]))

In [ ]:
def all_one_masks(model):
    return [
        torch.ones_like(layer.weight, dtype=torch.bool).cpu()
        for layer in prunable_layers(model)
    ]

def masks_at_ratios(model, ratios, selected_ratios):
    masks = all_one_masks(model)
    layers = prunable_layers(model)
    for index, ratio in enumerate(selected_ratios):
        masks[index] = mask_topk(layers[index].weight, float(ratio)).detach().cpu()
    return masks

def mask_topk(weight, fraction):
    flat = weight.detach().abs().flatten()
    keep = max(1, min(flat.numel(), int(round((1.0 - float(fraction)) * flat.numel()))))
    order = torch.argsort(flat, descending=True, stable=True)
    mask = torch.zeros_like(flat, dtype=torch.bool)
    mask[order[:keep]] = True
    return mask.reshape_as(weight)

def measure_mask(model, masks, calibration_loader, test_loader,
                 dense_calib, dense_test, device):
    masked = copy.deepcopy(model).to(device)
    apply_masks(masked, masks)
    masked_calib = logits(masked, calibration_loader, device)
    masked_test = logits(masked, test_loader, device)
    calib_error = (masked_calib - dense_calib).flatten(1).square().mean(dim=1)
    test_error = (masked_test - dense_test).flatten(1).square().mean(dim=1)
    result = {
        'calib_mean_distortion': float(calib_error.mean()),
        'test_mean_distortion': float(test_error.mean()),
        'calib_max_distortion': float(calib_error.max()),
        'test_max_distortion': float(test_error.max()),
    }
    del masked, masked_calib, masked_test
    if device.type == 'cuda':
        torch.cuda.empty_cache()
    return result

def measure_calib_mean(model, masks, calibration_loader, dense_calib, device):
    masked = copy.deepcopy(model).to(device)
    apply_masks(masked, masks)
    masked_calib = logits(masked, calibration_loader, device)
    error = (masked_calib - dense_calib).flatten(1).square().mean(dim=1)
    value = float(error.mean())
    del masked, masked_calib
    if device.type == 'cuda':
        torch.cuda.empty_cache()
    return value

def estimate_flops(model, masks, loader, device):
    records = []
    handles = []

    def hook(module, inputs, output):
        if isinstance(module, nn.Conv2d):
            output_elements = int(output.numel())
            kernel_ops = (module.in_channels // module.groups) * int(np.prod(module.kernel_size))
            dense_ops = output_elements * kernel_ops
        elif isinstance(module, nn.Linear):
            dense_ops = int(output.numel()) * module.in_features
        else:
            dense_ops = 0
        active_fraction = float(masks[len(records)].float().mean())
        records.append((dense_ops, dense_ops * active_fraction))

    for layer in prunable_layers(model):
        handles.append(layer.register_forward_hook(hook))

    model.eval()
    x, _ = next(iter(loader))
    with torch.no_grad():
        model(x.to(device))
    for handle in handles:
        handle.remove()

    dense_flops = sum(x[0] for x in records)
    sparse_flops = sum(x[1] for x in records)
    return {
        'dense_flops_per_batch': float(dense_flops),
        'sparse_flops_per_batch': float(sparse_flops),
        'flops_retained_fraction': float(sparse_flops / max(dense_flops, 1.0)),
        'flops_reduction_fraction': float(1.0 - sparse_flops / max(dense_flops, 1.0)),
    }

def jsonable(value):
    if isinstance(value, (np.integer,)):
        return int(value)
    if isinstance(value, (np.floating,)):
        return float(value)
    if isinstance(value, (np.ndarray,)):
        return value.tolist()
    if isinstance(value, (np.bool_,)):
        return bool(value)
    return value

In [ ]:
def prepare_allocations(model, calibration_loader, dense_calib, device, seed):
    started = time.monotonic()

    ratios, worst_curves = build_rdp_curves(
        model, calibration_loader, device,
        curve_points=CURVE_POINTS,
        max_pruning=1.0,
        worst_case=True,
        monotonic_repair=True,
    )
    mean_ratios, mean_curves = build_rdp_curves(
        model, calibration_loader, device,
        curve_points=CURVE_POINTS,
        max_pruning=1.0,
        worst_case=False,
        monotonic_repair=False,
    )
    if not np.allclose(ratios, mean_ratios):
        raise RuntimeError('RDP curve grids do not match.')

    reference_index = int(np.argmin(np.abs(ratios - PAIR_REFERENCE_RATIO)))
    pair_rows = []
    pair_interactions = []
    for layer_a, layer_b in pair_indices:
        masks_ab = all_one_masks(model)
        layers = prunable_layers(model)
        masks_ab[layer_a] = mask_topk(
            layers[layer_a].weight, PAIR_REFERENCE_RATIO
        ).detach().cpu()
        masks_ab[layer_b] = mask_topk(
            layers[layer_b].weight, PAIR_REFERENCE_RATIO
        ).detach().cpu()

        single_a = float(mean_curves[layer_a, reference_index])
        single_b = float(mean_curves[layer_b, reference_index])
        joint = measure_calib_mean(model, masks_ab, calibration_loader, dense_calib, device)
        interaction = joint - single_a - single_b
        row = {
            'layer_a': layer_a,
            'layer_a_name': layer_names[layer_a],
            'layer_b': layer_b,
            'layer_b_name': layer_names[layer_b],
            'reference_ratio': PAIR_REFERENCE_RATIO,
            'single_a': single_a,
            'single_b': single_b,
            'joint': joint,
            'interaction': interaction,
        }
        pair_rows.append(row)
        pair_interactions.append((layer_a, layer_b, interaction))

    corrected_curves_by_shrink = {}
    for shrinkage in PAIR_SHRINKAGE_FACTORS:
        corrected = mean_curves.copy()
        for layer_a, layer_b, interaction in pair_interactions:
            for layer_index in [layer_a, layer_b]:
                scale = mean_curves[layer_index] / max(
                    float(mean_curves[layer_index, reference_index]), 1e-12
                )
                corrected[layer_index] += (
                    float(shrinkage) * interaction / 2.0 * scale
                )
        corrected_curves_by_shrink[float(shrinkage)] = np.maximum(corrected, 0.0)

    rdhp_ratios, rdhp_curves, fishers = build_rdhp_curves(
        model, calibration_loader, device,
        curve_points=CURVE_POINTS,
        max_pruning=1.0,
        fisher_probes=FISHER_PROBES,
        fisher_damping=FISHER_DAMPING,
        fisher_mode='diagonal',
        dense_limit=2048,
        seed=seed,
    )
    if not np.allclose(ratios, rdhp_ratios):
        raise RuntimeError('RDP and RDHP curve grids do not match.')

    allocations = {}
    for target_sparsity in TARGET_SPARSITIES:
        additive_masks, additive_info = allocate_dp(
            model, ratios, worst_curves, target_sparsity, budget_bins=BUDGET_BINS
        )
        direct_masks, direct_info = allocate_dp(
            model, ratios, mean_curves, target_sparsity, budget_bins=BUDGET_BINS
        )
        rdhp_masks, rdhp_info = allocate_dp(
            model, ratios, rdhp_curves, target_sparsity, budget_bins=BUDGET_BINS
        )
        methods = {
            'additive_rdp': (additive_masks, additive_info),
            'direct_rdp': (direct_masks, direct_info),
            'rdhp': (rdhp_masks, rdhp_info),
        }
        for shrinkage, corrected_curves in corrected_curves_by_shrink.items():
            masks, info = allocate_dp(
                model, ratios, corrected_curves, target_sparsity,
                budget_bins=BUDGET_BINS
            )
            methods[f'pair_corr_lambda_{shrinkage:.2f}'] = (masks, info)
        allocations[float(target_sparsity)] = methods
        if device.type == 'cuda':
            torch.cuda.empty_cache()

    return {
        'ratios': ratios,
        'worst_curves': worst_curves,
        'mean_curves': mean_curves,
        'rdhp_curves': rdhp_curves,
        'corrected_curves_by_shrink': corrected_curves_by_shrink,
        'pair_rows': pair_rows,
        'allocations': allocations,
        'elapsed_seconds': time.monotonic() - started,
    }

In [ ]:
def run_seed(seed):
    started = time.monotonic()
    seed_everything(seed)
    device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')

    run_root = DRIVE_ROOT / f'rdhp_rdp_runs_seed_{seed}'
    checkpoint = run_root / 'dense_resnet32.pt'
    output_dir = OUTPUT_ROOT / f'seed_{seed}'
    output_dir.mkdir(parents=True, exist_ok=True)
    if not checkpoint.exists():
        raise FileNotFoundError(f'Missing checkpoint: {checkpoint}')

    loader_args = Namespace(
        synthetic=False,
        data_root=str(DATA_ROOT),
        download=False,
        train_samples=TRAIN_SAMPLES,
        calib_samples=CALIB_SAMPLES,
        test_samples=TEST_SAMPLES,
        batch_size=BATCH_SIZE,
        seed=seed,
    )
    (train_loader, calibration_loader, test_loader), split_info = make_loaders(loader_args)

    model = load_resnet32().to(device)
    checkpoint_sha256 = load_checkpoint(model, checkpoint)
    dense_calib = logits(model, calibration_loader, device)
    dense_test = logits(model, test_loader, device)
    dense_accuracy = accuracy(model, test_loader, device)
    dense_flops = estimate_flops(model, all_one_masks(model), test_loader, device)

    prepared = prepare_allocations(model, calibration_loader, dense_calib, device, seed)
    pd.DataFrame(prepared['pair_rows']).to_csv(
        output_dir / 'pair_interactions.csv', index=False
    )

    measured_rows = []
    fine_tune_rows = []
    dense_control_rows = []

    for target_sparsity, methods in prepared['allocations'].items():
        for method, (masks, allocation_info) in methods.items():
            values = measure_mask(
                model, masks, calibration_loader, test_loader,
                dense_calib, dense_test, device
            )
            masked = copy.deepcopy(model).to(device)
            apply_masks(masked, masks)
            immediate_accuracy = accuracy(masked, test_loader, device)
            stats = model_stats(masked)
            flops = estimate_flops(masked, masks, test_loader, device)

            measured_rows.append({
                'seed': seed,
                'target_sparsity': target_sparsity,
                'method': method,
                'calib_mean_distortion': values['calib_mean_distortion'],
                'test_mean_distortion': values['test_mean_distortion'],
                'calib_max_distortion': values['calib_max_distortion'],
                'test_max_distortion': values['test_max_distortion'],
                'immediate_accuracy': immediate_accuracy,
                'dense_checkpoint_accuracy': dense_accuracy,
                'accuracy_loss_immediate_vs_checkpoint': immediate_accuracy - dense_accuracy,
                'parameters': stats['parameters'],
                'nonzero': stats['nonzero'],
                'sparsity_actual': stats['sparsity'],
                'retained_fraction': stats['retained_fraction'],
                **flops,
                'allocation_achieved_sparsity': allocation_info['achieved_sparsity'],
            })
            del masked
            if device.type == 'cuda':
                torch.cuda.empty_cache()

    measured_df = pd.DataFrame(measured_rows)
    oracle_by_target = measured_df.groupby('target_sparsity')['test_mean_distortion'].min()
    measured_df['test_distortion_oracle'] = measured_df['target_sparsity'].map(oracle_by_target)
    measured_df['test_distortion_regret'] = (
        measured_df['test_mean_distortion'] - measured_df['test_distortion_oracle']
    ) / measured_df['test_distortion_oracle'].clip(lower=1e-12)

    for replicate, offset in enumerate(FINETUNE_SEED_OFFSETS):
        for epochs in FINETUNE_EPOCHS:
            dense_model = copy.deepcopy(model).to(device)
            train_epochs(
                dense_model, train_loader, device, epochs,
                LEARNING_RATE, seed + offset + epochs
            )
            dense_acc = accuracy(dense_model, test_loader, device)
            dense_control_rows.append({
                'seed': seed,
                'method': 'dense_baseline',
                'target_sparsity': 0.0,
                'replicate': replicate,
                'fine_tune_seed_offset': offset,
                'epochs': epochs,
                'accuracy': dense_acc,
                'checkpoint_accuracy': dense_accuracy,
                'accuracy_loss_vs_checkpoint': dense_acc - dense_accuracy,
            })
            del dense_model
            if device.type == 'cuda':
                torch.cuda.empty_cache()

    for _, row in measured_df[measured_df['method'].isin(PRIMARY_METHODS)].iterrows():
        target_sparsity = float(row['target_sparsity'])
        method = row['method']
        method_masks, _ = prepared['allocations'][target_sparsity][method]
        for replicate, offset in enumerate(FINETUNE_SEED_OFFSETS):
            for epochs in FINETUNE_EPOCHS:
                finetuned = copy.deepcopy(model).to(device)
                apply_masks(finetuned, method_masks)
                train_epochs(
                    finetuned, train_loader, device, epochs,
                    LEARNING_RATE, seed + offset + epochs,
                    masks=method_masks
                )
                apply_masks(finetuned, method_masks)
                final_accuracy = accuracy(finetuned, test_loader, device)
                fine_tune_rows.append({
                    'seed': seed,
                    'method': method,
                    'target_sparsity': target_sparsity,
                    'replicate': replicate,
                    'fine_tune_seed_offset': offset,
                    'epochs': epochs,
                    'accuracy': final_accuracy,
                    'checkpoint_accuracy': dense_accuracy,
                    'accuracy_loss_vs_checkpoint': final_accuracy - dense_accuracy,
                })
                del finetuned
                if device.type == 'cuda':
                    torch.cuda.empty_cache()

    dense_df = pd.DataFrame(dense_control_rows)
    fine_df = pd.DataFrame(fine_tune_rows)
    dense_means = dense_df.groupby('epochs')['accuracy'].mean().to_dict()
    dense_stds = dense_df.groupby('epochs')['accuracy'].std(ddof=1).to_dict()

    primary_rows = []
    for _, row in measured_df[measured_df['method'].isin(PRIMARY_METHODS)].iterrows():
        output = row.to_dict()
        for epochs in FINETUNE_EPOCHS:
            values = fine_df[
                (fine_df['method'] == row['method']) &
                (fine_df['target_sparsity'] == row['target_sparsity']) &
                (fine_df['epochs'] == epochs)
            ]['accuracy']
            output[f'finetuned_accuracy_{epochs}_mean'] = float(values.mean())
            output[f'finetuned_accuracy_{epochs}_std'] = float(values.std(ddof=1))
            output[f'dense_accuracy_{epochs}_mean'] = float(dense_means[epochs])
            output[f'accuracy_loss_vs_finetuned_dense_{epochs}'] = (
                float(values.mean()) - float(dense_means[epochs])
            )
        primary_rows.append(output)

    pair_df = measured_df[
        measured_df['method'].str.startswith('pair_corr_lambda_')
    ].copy()
    pair_df.to_csv(output_dir / 'pair_ablation.csv', index=False)
    measured_df.to_csv(output_dir / 'all_method_distortion_accuracy.csv', index=False)
    pd.DataFrame(primary_rows).to_csv(output_dir / 'primary_method_results.csv', index=False)
    dense_df.to_csv(output_dir / 'dense_finetune_replicates.csv', index=False)
    fine_df.to_csv(output_dir / 'primary_finetune_replicates.csv', index=False)

    np.savez_compressed(
        output_dir / 'allocation_curves.npz',
        ratios=prepared['ratios'],
        worst_curves=prepared['worst_curves'],
        mean_curves=prepared['mean_curves'],
        rdhp_curves=prepared['rdhp_curves'],
        **{
            f'pair_corr_lambda_{shrinkage:.2f}': curves
            for shrinkage, curves in prepared['corrected_curves_by_shrink'].items()
        },
    )

    fig, axes = plt.subplots(1, 2, figsize=(15, 5))
    primary_plot_df = pd.DataFrame(primary_rows)
    for target_sparsity in TARGET_SPARSITIES:
        distortion_plot_df = measured_df[
            (measured_df['target_sparsity'] == target_sparsity) &
            (measured_df['method'].isin(PRIMARY_METHODS))
        ]
        accuracy_plot_df = primary_plot_df[
            (primary_plot_df['target_sparsity'] == target_sparsity) &
            (primary_plot_df['method'].isin(PRIMARY_METHODS))
        ]
        axes[0].plot(
            distortion_plot_df['method'], distortion_plot_df['test_mean_distortion'],
            marker='o', label=f'{target_sparsity:.0%} sparsity'
        )
        axes[1].plot(
            accuracy_plot_df['method'], accuracy_plot_df['finetuned_accuracy_20_mean'],
            marker='o', label=f'{target_sparsity:.0%} sparsity'
        )
    axes[0].set_ylabel('Test mean logit distortion')
    axes[0].set_title(f'Seed {seed}: primary distortion')
    axes[0].legend()
    axes[1].set_ylabel('20-epoch test accuracy (%)')
    axes[1].set_title(f'Seed {seed}: primary accuracy')
    axes[1].legend()
    fig.tight_layout()
    fig.savefig(output_dir / 'fair_accuracy_comparison.png', dpi=160)
    plt.close(fig)

    print(
        f'Seed {seed} finished in {time.monotonic() - started:.1f}s | '
        f'dense checkpoint accuracy={dense_accuracy:.2f}% | '
        f'primary rows={len(primary_rows)}'
    )
    return pd.DataFrame(primary_rows), pair_df, fine_df, dense_df

all_primary = []
all_pair = []
all_fine = []
all_dense = []
for seed in SEEDS:
    primary_df, pair_df, fine_df, dense_df = run_seed(seed)
    all_primary.append(primary_df)
    all_pair.append(pair_df)
    all_fine.append(fine_df)
    all_dense.append(dense_df)

primary_results = pd.concat(all_primary, ignore_index=True)
pair_results = pd.concat(all_pair, ignore_index=True)
fine_results = pd.concat(all_fine, ignore_index=True)
dense_results = pd.concat(all_dense, ignore_index=True)

primary_results.to_csv(OUTPUT_ROOT / 'all_seeds_primary_results.csv', index=False)
pair_results.to_csv(OUTPUT_ROOT / 'all_seeds_pair_ablation.csv', index=False)
fine_results.to_csv(OUTPUT_ROOT / 'all_seeds_primary_finetune_replicates.csv', index=False)
dense_results.to_csv(OUTPUT_ROOT / 'all_seeds_dense_finetune_replicates.csv', index=False)

aggregate_columns = [
    'test_mean_distortion',
    'test_distortion_regret',
    'immediate_accuracy',
    'finetuned_accuracy_5_mean',
    'finetuned_accuracy_20_mean',
    'accuracy_loss_vs_finetuned_dense_5',
    'accuracy_loss_vs_finetuned_dense_20',
    'sparsity_actual',
    'flops_reduction_fraction',
]
aggregate = primary_results.groupby(['target_sparsity', 'method'])[aggregate_columns].agg(['mean', 'std'])
aggregate.to_csv(OUTPUT_ROOT / 'all_seeds_primary_mean_std.csv')
display(aggregate)

pair_aggregate = pair_results.groupby(
    ['target_sparsity', 'method']
)[[
    'test_mean_distortion',
    'test_distortion_regret',
    'immediate_accuracy',
]].agg(['mean', 'std'])
pair_aggregate.to_csv(OUTPUT_ROOT / 'all_seeds_pair_ablation_mean_std.csv')
display(pair_aggregate)

summary = {
    'seeds': SEEDS,
    'target_sparsities': TARGET_SPARSITIES,
    'calibration_samples': CALIB_SAMPLES,
    'test_samples': TEST_SAMPLES,
    'curve_points': CURVE_POINTS,
    'fisher_probes': FISHER_PROBES,
    'finetune_epochs': FINETUNE_EPOCHS,
    'finetune_seed_offsets': FINETUNE_SEED_OFFSETS,
    'pair_shrinkage_factors': PAIR_SHRINKAGE_FACTORS,
    'primary_methods': PRIMARY_METHODS,
    'primary_results': primary_results.to_dict(orient='records'),
    'pair_ablation_results': pair_results.to_dict(orient='records'),
    'dense_finetune_replicates': dense_results.to_dict(orient='records'),
}
(OUTPUT_ROOT / 'all_seeds_summary.json').write_text(
    json.dumps(summary, indent=2, default=jsonable) + '\n',
    encoding='utf-8',
)
print('Saved fair accuracy ablation results to:', OUTPUT_ROOT)

## Publish results

Results are copied to `results/rdhp_rdp_fair_accuracy_ablation_all_seeds`.

In [ ]:
if not PUBLISH_RESULTS:
    print('Publishing disabled.')
else:
    from google.colab import userdata

    write_token = userdata.get('GITHUB_WRITE_TOKEN')
    if not write_token:
        raise RuntimeError('Add GITHUB_WRITE_TOKEN with Contents read/write access.')

    upload_repo = Path('/content/rdhp-rdp-fair-accuracy-upload')
    if upload_repo.exists():
        shutil.rmtree(upload_repo)

    askpass = Path('/tmp/github-askpass-full-network-write.sh')
    askpass.write_text(
        '#!/bin/sh\n'
        'case "$1" in\n'
        "  *Username*) printf '%s\\n' x-access-token ;;\n"
        "  *Password*) printf '%s\\n' \"$GITHUB_WRITE_TOKEN\" ;;\n"
        "  *) printf '\\n' ;;\n"
        'esac\n'
    )
    askpass.chmod(0o700)

    upload_env = os.environ.copy()
    upload_env.update({
        'GIT_ASKPASS': str(askpass),
        'GIT_TERMINAL_PROMPT': '0',
        'GITHUB_WRITE_TOKEN': write_token,
    })

    try:
        subprocess.run([
            'git', 'clone',
            'https://github.com/hamidrg20001379/rdhp-rdp-colab-experiment.git',
            str(upload_repo),
        ], env=upload_env, check=True)

        destination = upload_repo / 'results' / 'rdhp_rdp_full_network_corrected_all_seeds'
        shutil.copytree(OUTPUT_ROOT, destination, dirs_exist_ok=True)

        subprocess.run(['git', '-C', str(upload_repo), 'config', 'user.name', 'Colab Results Bot'], check=True)
        subprocess.run(['git', '-C', str(upload_repo), 'config', 'user.email', 'colab-results@users.noreply.github.com'], check=True)
        subprocess.run(['git', '-C', str(upload_repo), 'add', 'results', '--'])
        changed = subprocess.run([
            'git', '-C', str(upload_repo), 'diff', '--cached', '--quiet'
        ]).returncode != 0

        if changed:
            subprocess.run([
                'git', '-C', str(upload_repo), 'commit', '-m',
                'Publish fair accuracy ablation results for seeds 42-44',
            ], check=True)
            subprocess.run([
                'git', '-C', str(upload_repo), 'push', 'origin', 'HEAD:master',
            ], env=upload_env, check=True)
            print('Fair accuracy ablation results uploaded.')
        else:
            print('No new results to upload.')
    finally:
        askpass.unlink(missing_ok=True)